# 🍃 Módulo 3: Clasificación de Patologías en Hojas por Visión Artificial
## Proyecto: AgroSmart Perú — Universidad / Software Inteligente

**Objetivo:** Clasificar patologías foliares provocadas por las altas temperaturas y exceso de humedad de El Niño Costero (*Tizón bacteriano - Xanthomonas* y *Mancha marrón - Bipolaris* frente a *Hoja sana*).  
**Salida:** Diagnóstico patológico con probabilidad de certidumbre y recomendación técnica aprobada por SENASA.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sample_dir = '../data/sample_leaves'
print("Carpeta de muestras:", os.path.abspath(sample_dir))


### 1. Inspección Visual del Banco de Muestras Foliares


In [ ]:
clases = ['hoja_sana.jpg', 'tizon_bacteriano.jpg', 'mancha_marron.jpg']

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, c in enumerate(clases):
    path = os.path.join(sample_dir, c)
    img = Image.open(path)
    axes[i].imshow(img)
    axes[i].set_title(c.replace('.jpg', '').replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].axis('off')
plt.tight_layout()
plt.show()


### 2. Extracción de Descriptores Cromáticos y Algoritmo de Inferencia


In [ ]:
def clasificar_hoja(image_path):
    img = Image.open(image_path).convert('RGB').resize((150, 150))
    arr = np.array(img, dtype=np.float32) / 255.0
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    
    # Índices fisiológicos
    greenness = float(np.mean(g - (r + b) / 2.0))
    brownness = float(np.mean(r * 0.6 + g * 0.3 - b * 0.9))
    yellow_halo = float(np.mean((r + g) / 2.0 - b))
    
    if greenness > 0.08 and brownness < 0.12:
        prob_sana = min(0.96, 0.75 + greenness * 1.5)
        prob_tizon = (1.0 - prob_sana) * 0.6
        prob_mancha = (1.0 - prob_sana) * 0.4
    elif yellow_halo > 0.20 or (brownness > 0.15 and greenness < 0.02):
        prob_tizon = min(0.94, 0.70 + yellow_halo * 0.8)
        prob_mancha = (1.0 - prob_tizon) * 0.7
        prob_sana = (1.0 - prob_tizon) * 0.3
    else:
        prob_mancha = min(0.92, 0.68 + brownness * 1.2)
        prob_tizon = (1.0 - prob_mancha) * 0.65
        prob_sana = (1.0 - prob_mancha) * 0.35
        
    probs = {'Hoja Sana': prob_sana, 'Tizón Bacteriano': prob_tizon, 'Mancha Marrón': prob_mancha}
    best_clase = max(probs, key=probs.get)
    return best_clase, probs[best_clase] * 100, probs

print("Función de inferencia compilada.")


### 3. Evaluación de Rendimiento sobre el Banco de Muestras


In [ ]:
for c in clases:
    p = os.path.join(sample_dir, c)
    clase_pred, conf, dist = clasificar_hoja(p)
    print(f"Archivo: {c:20} ➡️ Predicción: {clase_pred:20} (Confianza: {conf:.1f}%)")


### 4. Directrices Técnicas Asociadas (SENASA)
* **Hoja Sana:** Mantener monitoreo preventivo y descolmatación de drenes.
* **Tizón Bacteriano (*Xanthomonas*):** Aplicar bactericidas a base de cobre y suspender fertilización con nitrógeno.
* **Mancha Marrón (*Bipolaris*):** Drenar parcelas encharcadas y aplicar fungicidas sistémicos (Triazoles).
